In [1]:
import numpy as np
import xlwings as xw
from sklearn.linear_model import LinearRegression
import os
from glob import glob   
import pandas as pd
import numpy.linalg as la


# Input parameters
OXY  = 210.0
# Q is PPFD
Q    = 1200.0
abs_val = 0.93
Rdfixed = False
Rdfixed2= False

## Set your folders
names = ['raw20190627SY','raw20190709SY','raw20190728SY','raw20190803SY','raw20190810SY','raw20190818SY','raw20190823SY','raw20190907SY','raw20200615SY']

## Batch
for name in names:
    input = 'photosynthesis/data/soybean2019/%s' % name
    output = 'photosynthesis/data/DD_01/Bernacchi_2003%s' % name
  
   
    os.makedirs(output,exist_ok=True)
    files = glob('%s/*.xlsx' % input)
    files.sort()
    file_names = []
    Vcmaxs = []
    Rd1s = []
    Vcmax25s = []
    R21s = []
    RMSE1_list = [] 
    GammaS_list = []
 
    for file in files:
        print(file)
        ## Read raw data
        xlsx = pd.read_excel(file,skiprows=14)
        A = np.array(xlsx['A'][1:],np.float32)
        T = np.array(xlsx['Tleaf'][1:],np.float32)    # [K]
        Ci = np.array(xlsx['Ci'][1:],np.float32)   # [Pa]
        ## Set criteria to remove points
        msk = np.full(A.size,True)

        A = np.array(A[msk])
        Ci = np.array(Ci[msk])   
        T = np.array(T[msk])
        ## Sort from small Ci to large Ci
        idx = np.argsort(Ci)
        A = A[idx]
        Ci = Ci[idx]
        Tleaf = T[idx]
        condition1 = Ci < 200
        condition2 = Ci > 300

        Tleaf1= Tleaf[condition1]
        A1    = A[condition1]
        Ci1   = Ci[Ci < 200]

        Tleaf2= Tleaf[condition2]
        A2    = A[condition2]
        Ci2   = Ci[Ci > 300]

        # Most likely are the intrinsic parameters
        tau_c=-14.37
        tau_d=-37.83
        gc=0.008314
        Vm_c=26.35
        Vm_d=65.33
        Kc_c=38.05
        Kc_d=79.43
        Ko_c=20.3
        Ko_d=36.38
        beta=0.5
        # Use list here in order to pass reference in function
        ### Do not change the following three list to three scalar###
        FixedRd = [0.0]
        FixedRd2= [0.0]
        FixedRd3= [0.0]
      

        Tleaf1 = np.array(Tleaf1)
        A1     = np.array(A1)
        Ci1    = np.array(Ci1)

        Tleaf2 = np.array(Tleaf2)
        A2     = np.array(A2)
        Ci2    = np.array(Ci2)

        Temp      = 0.5*( np.average(Tleaf1) + np.average(Tleaf2) )
        Rcorr     = np.exp(18.72-46.39/(gc*(Temp+273.15)))
        f         = abs_val*(-30.74+0.206*(Temp+273.15)-0.000337*(Temp+273.15)*(Temp+273.15))*beta
        th        =-31.99+0.222*(273.15+Temp)-0.000375*(273.15+Temp)*(273.15+Temp)
        Jcorrtemp = np.exp(17.57-43.54/(gc*(Temp+273.15)))

        

        def call_LINEST(x_values,y_values,cal_intercept):
           
            app = xw.App(visible=False)
            wb = app.books.add()  
            sheet = wb.sheets[0]  
            data_len = len(x_values)

            sheet.range("A1").value = "X"
            sheet.range("A2").options(transpose=True).value = x_values
            sheet.range("B1").value = "Y"
            sheet.range("B2").options(transpose=True).value = y_values

            if(cal_intercept):
                sheet.range("D1:E5").formula_array = "="+"LINEST($B$2:$B$"+str(data_len+1)+", $A$2:$A$"+str(data_len+1)+", True, TRUE)"
            else:
                sheet.range("D1:E5").formula_array = "="+"LINEST($B$2:$B$"+str(data_len+1)+", $A$2:$A$"+str(data_len+1)+", False, TRUE)"

            linest_output = sheet.range("D1:E4").value
            slope = linest_output[0][0]
            intercept = linest_output[0][1]
            slope_std = linest_output[1][0] 
            intercept_std = linest_output[1][1] 

            wb.close()
            app.quit()
            return slope, slope_std, intercept, intercept_std
        
        
        def cal_Vcmax(Tleaf,A,Ci):
            GammaS = (0.5 * OXY) / np.exp(tau_c - tau_d / (gc * (Tleaf + 273.15)))
            fprime1=(1 - GammaS / Ci) * (Ci * np.exp(Vm_c - Vm_d / (gc * (Tleaf + 273.15)))) / (Ci + np.exp(Kc_c - Kc_d / (gc * (Tleaf + 273.15))) * (1 + OXY / np.exp(Ko_c - Ko_d / (gc * (Tleaf + 273.15)))))
            if(Rdfixed):
                Aprime1=A - FixedRd[0]
            else:
                Aprime1=A
            
            fi2 = fprime1
            Ai2 = Aprime1
            whether_fit_intercept = not Rdfixed
            slope, slope_std, intercept, intercept_std = call_LINEST(fi2,Ai2,whether_fit_intercept)
            
            if(Rdfixed2):
                FixedRd2[0] = intercept
            else:
                FixedRd2[0] = 0.0
            FixedRd3[0] = FixedRd2[0]*Rcorr
            # Calculate predicted values using the linear model
            predicted_Ai2 = slope * fi2 + intercept
            # Calculate R2
            ss_res = np.sum((Ai2 - predicted_Ai2) ** 2)  # Sum of squares of residuals
            ss_tot = np.sum((Ai2 - np.mean(Ai2)) ** 2)  # Total sum of squares
            R21 = 1 - (ss_res / ss_tot)  # R2 value
            
            return slope,slope_std,intercept,intercept_std,fprime1,Aprime1,R21,predicted_Ai2,GammaS
    
        plus_minus_sign = "\u00B1"
        Vcmax, Vcmax_std, Rd1, Rd_std1, fprime1, Aprime1,R21,predicted_Ai2,GammaS = cal_Vcmax(Tleaf1,A1,Ci1)
        # ===== Rd =====
        if (Rd1 is None) or np.isnan(Rd1):
            print(f"⚠: {file}")
            continue

        if Rd1 > 0:
            print(f"⚠ Rd ≤ 0: {file}")
            Rd1 = 0.0
            FixedRd[0] = Rd1  
            Rdfixed = True
            Vcmax, Vcmax_std, Rd1_refit, Rd_std1, fprime1, Aprime1, R21, predicted_Ai2,GammaS = cal_Vcmax(Tleaf1, A1, Ci1)
            Rd1 = 0.0
            Rdfixed = False

        Vcmax25 = np.mean(Vcmax / np.exp(Vm_c-Vm_d/(gc*(Tleaf1+273.15))))
        RMSE1 = np.sqrt(np.mean((A1 - predicted_Ai2) ** 2))
        Rd1 =-Rd1
        file_names.append(file.split('\\')[-1]+',')
        Vcmax25s.append(Vcmax25) 
        Vcmaxs.append(Vcmax) 
        Rd1s.append(Rd1)
        GammaS_list.append(float(np.mean(GammaS)))
        R21s.append(R21)
        RMSE1_list.append(RMSE1)

    df0 = pd.DataFrame(file_names, columns=['filenames'])
    df1 = pd.DataFrame(Vcmax25s, columns=['Vcmax25'])
    df2 = pd.DataFrame(Vcmaxs, columns=['Vcmax'])
    df3 = pd.DataFrame(Rd1s, columns=['Rd'])
    df4 = pd.DataFrame(R21s, columns=['R2'])
    df5 = pd.DataFrame(GammaS_list, columns=['GammaS'])
    df6 = pd.DataFrame(RMSE1_list, columns=['RMSE'])
    df_combined = pd.concat([df0,df1,df2,df3,df5,df4,df6], axis=1)

    filename = output + '/result.csv'
    # print(filename)
    df_combined.to_csv(filename,index=False)    

 
    

D:/博士学习资料/根据Vcmax25计算A/大豆01/raw20200615SY\2020-06-15-1247_ACi-top2.xlsx
⚠ Rd ≤ 0，强制设为 0，并重新拟合 Vcmax: D:/博士学习资料/根据Vcmax25计算A/大豆01/raw20200615SY\2020-06-15-1247_ACi-top2.xlsx
D:/博士学习资料/根据Vcmax25计算A/大豆01/raw20200615SY\2020-06-15-1321_ACi-soy-top3.xlsx
⚠ Rd ≤ 0，强制设为 0，并重新拟合 Vcmax: D:/博士学习资料/根据Vcmax25计算A/大豆01/raw20200615SY\2020-06-15-1321_ACi-soy-top3.xlsx
D:/博士学习资料/根据Vcmax25计算A/大豆01/raw20200615SY\2020-06-15-1356_ACi-soy-top4.xlsx
D:/博士学习资料/根据Vcmax25计算A/大豆01/raw20200615SY\2020-06-15-1443_ACi-soy-top5.xlsx
⚠ Rd ≤ 0，强制设为 0，并重新拟合 Vcmax: D:/博士学习资料/根据Vcmax25计算A/大豆01/raw20200615SY\2020-06-15-1443_ACi-soy-top5.xlsx
